In [72]:
#Decision tree Clasifier
import numpy as np

In [89]:
#Step 1: Encode the Dataset
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
data_array = np.array(data)
label={"Beer":0,"Wine":1,"Whiskey":2}
X=data_array[:,0:3].astype(float)
y_raw=data_array[:,3]
Y=np.array([label[name] for name in y_raw])

In [90]:
#Step 2: Implement Gini Impurity
def Gini_impurity(y):
    if len(y)==0:
        return 0
    _,counts=np.unique(y,return_counts=True)
    prob=counts/len(y)
    gini=1-np.sum(prob**2)
    return gini

In [91]:
#seeing which 
def get_best_split(X,Y):
    best_gini=float('inf')
    best_split={}
    no_samples,no_features=X.shape
    for feat_idx in range(no_features): 
        thresholds=np.unique(X[:,feat_idx])
        for threshold in thresholds:
            #now selecting a particular threshold and splitting accordingly
            left=X[:,feat_idx]<=threshold
            right=X[:,feat_idx]>threshold
            if len(Y[left])==0 or len(Y[right])==0:
                continue
            right_gini=Gini_impurity(Y[right])
            left_gini=Gini_impurity(Y[left])
            current_gini=(len(Y[left])*left_gini+len(Y[right])*right_gini)/no_samples
            if current_gini<best_gini:
                #update 
                best_gini = current_gini
                best_split={'feature_idx':feat_idx,'threshold':threshold,'left_x':X[left],'left_y':Y[left],'right_x':X[right],'right_y':Y[right]}
    return best_split

In [92]:
class Node:
    def __init__(self,feature_idx=None,threshold=None,left=None,right=None,value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value 
    

In [93]:
def build_tree(X,Y,depth=0,max_depth=10):
        no_samples,no_features=X.shape
        unique_labels=np.unique(Y)
        if len(unique_labels)==1 or depth>=max_depth or no_samples<2:
            vals,counts=np.unique(Y,return_counts=True)
            most_common= vals[np.argmax(counts)]
            return Node(value=most_common)
        best_split=get_best_split(X,Y)
        if not best_split:
            vals, counts = np.unique(Y, return_counts=True)
            return Node(value=vals[np.argmax(counts)])
        left_subtree=build_tree(best_split['left_x'],best_split['left_y'],depth+1,max_depth)
        right_subtree=build_tree(best_split['right_x'],best_split['right_y'],depth+1,max_depth)
        return Node(
        feature_idx=best_split['feature_idx'], 
        threshold=best_split['threshold'], 
        left=left_subtree, 
        right=right_subtree
    )

In [94]:
#now in this step we use the above generated tree to classify new drinks
def predict_drink(node,x):
    if node.value is not None:
        return node.value
    #taking feature of x on which we decide 
    feature_val=x[node.feature_idx]
    if feature_val<=node.threshold:
        return predict_drink(node.left,x)
    else:
        return predict_drink(node.right,x)

In [95]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
node=build_tree(X,Y,0,10)
predicted=[predict_drink(node, x) for x in test_data]
print(predicted)

[np.int64(1), np.int64(2), np.int64(1)]
